In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression,SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

current_dir = Path.cwd()
project_root = current_dir.parents[2]

classification_models = {
    "decision_tree": DecisionTreeClassifier(random_state=42),

    "random_forest": RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),

    "extra_trees": ExtraTreesClassifier(
        random_state=42,
        n_jobs=-1
    ),

    "xgboost": XGBClassifier(
    tree_method="hist",
    device="cuda",
    eval_metric="logloss"
    ),

    "adaboost": AdaBoostClassifier(
        algorithm="SAMME",
        random_state=42
    ),

    "logistic_regression": LogisticRegression(
        random_state=42,
        n_jobs=-1
    ),

    "sgd": SGDClassifier(
        loss="log_loss",
        random_state=42,
        n_jobs=-1
    ),

    "svc": SVC(),

    "knn": KNeighborsClassifier(
        n_jobs=-1
    ),

    "gaussian_nb": GaussianNB()
}

In [2]:
import json

with open(project_root/"SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/Domain_data.json", "r") as archivo:
    datos = json.load(archivo)

print(datos)


{'M_data': ['NP1COG_mean', 'NP1COG_min', 'NP1COG_max', 'NP1COG_var', 'NP1COG_std', 'NP1HALL_mean', 'NP1HALL_min', 'NP1HALL_max', 'NP1HALL_var', 'NP1HALL_std', 'NP1DPRS_mean', 'NP1DPRS_min', 'NP1DPRS_max', 'NP1DPRS_var', 'NP1DPRS_std', 'NP1ANXS_mean', 'NP1ANXS_min', 'NP1ANXS_max', 'NP1ANXS_var', 'NP1ANXS_std', 'NP1APAT_mean', 'NP1APAT_min', 'NP1APAT_max', 'NP1APAT_var', 'NP1APAT_std', 'NP1DDS_mean', 'NP1DDS_min', 'NP1DDS_max', 'NP1DDS_var', 'NP1DDS_std', 'NP1SLPN_mean', 'NP1SLPN_min', 'NP1SLPN_max', 'NP1SLPN_var', 'NP1SLPN_std', 'NP1SLPD_mean', 'NP1SLPD_min', 'NP1SLPD_max', 'NP1SLPD_var', 'NP1SLPD_std', 'NP1PAIN_mean', 'NP1PAIN_min', 'NP1PAIN_max', 'NP1PAIN_var', 'NP1PAIN_std', 'NP1URIN_mean', 'NP1URIN_min', 'NP1URIN_max', 'NP1URIN_var', 'NP1URIN_std', 'NP1CNST_mean', 'NP1CNST_min', 'NP1CNST_max', 'NP1CNST_var', 'NP1CNST_std', 'NP1LTHD_mean', 'NP1LTHD_min', 'NP1LTHD_max', 'NP1LTHD_var', 'NP1LTHD_std', 'NP1FATG_mean', 'NP1FATG_min', 'NP1FATG_max', 'NP1FATG_var', 'NP1FATG_std', 'NP2SPCH_m

In [3]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)


def evaluate_models_10x10_oof_and_test(
    X_df: pd.DataFrame,
    y_df: pd.DataFrame,
    models: dict,
    outer_splits: int = 10,
    inner_splits: int = 10,
    test_size: float = 0.2,
    random_state: int = 42,
    average: str = "macro",   
    decimals: int = 4,
):
    """
    Evalúa modelos con:
      - Outer: StratifiedShuffleSplit (n veces)
      - Inner: StratifiedKFold para OOF en train
      - Métricas: Accuracy, Precision, Recall, F1, AUC (multiclase OVR)
    AUC:
      - Se calcula SOLO si el estimador soporta predict_proba o decision_function
      - Si no, AUC = np.nan
    """

    y = y_df.iloc[:, 0].to_numpy()
    X = X_df.to_numpy()

    # AUC multiclass requiere scores (n_muestras, n_clases)
    def get_scores_multiclass(fitted_pipe: Pipeline, X_part: np.ndarray):
        """
        Devuelve scores continuos por clase:
          - predict_proba si existe y devuelve (n, n_classes)
          - si no, decision_function si existe y devuelve (n, n_classes)
          - si no, None
        """
        if hasattr(fitted_pipe, "predict_proba"):
            try:
                proba = fitted_pipe.predict_proba(X_part)
                if isinstance(proba, np.ndarray) and proba.ndim == 2:
                    return proba
            except Exception:
                pass

        if hasattr(fitted_pipe, "decision_function"):
            try:
                dec = fitted_pipe.decision_function(X_part)
                if isinstance(dec, np.ndarray) and dec.ndim == 2:
                    return dec
            except Exception:
                pass

        return None

    def auc_multiclass_ovr(y_true: np.ndarray, scores, average: str):
        """
        AUC multiclase con One-vs-Rest.
        Devuelve np.nan si no hay scores válidos o si roc_auc_score falla.
        """
        if scores is None:
            return np.nan
        try:
            return roc_auc_score(y_true, scores, multi_class="ovr", average=average)
        except Exception:
            return np.nan

    def compute_metrics(y_true, y_pred, auc_value=np.nan):
        return {
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, average=average, zero_division=0),
            "Recall": recall_score(y_true, y_pred, average=average, zero_division=0),
            "F1": f1_score(y_true, y_pred, average=average, zero_division=0),
            "AUC": auc_value,
        }

    def summarize(metrics_list, suffix: str):
        df = pd.DataFrame(metrics_list)
        # mean/std ignoran NaN -> ideal para AUC cuando no se pueda calcular
        mean = df.mean(numeric_only=True)
        std = df.std(ddof=1, numeric_only=True)
        return {
            f"{c}_{suffix}": f"{mean[c]:.{decimals}f} ± {std[c]:.{decimals}f}"
            for c in mean.index
        }

    outer = StratifiedShuffleSplit(
        n_splits=outer_splits, test_size=test_size, random_state=random_state
    )

    test_summary_rows = []
    cv_summary_rows = []

    for model_name, estimator in models.items():
        print(f"Evaluating {model_name}...")
        test_metrics_all = []
        cv_metrics_all = []

        for train_idx, test_idx in outer.split(X, y):
            X_train, y_train = X[train_idx], y[train_idx]
            X_test, y_test = X[test_idx], y[test_idx]

            # ---------- OOF en TRAIN (inner CV) ----------
            inner = StratifiedKFold(
                n_splits=inner_splits, shuffle=True, random_state=random_state
            )

            oof_pred = np.zeros_like(y_train)

            # Para AUC OOF necesitamos scores OOF (n_train, n_classes)
            oof_scores = None

            for tr_idx, val_idx in inner.split(X_train, y_train):
                pipe = Pipeline([
                    ("scaler", StandardScaler()),
                    ("model", clone(estimator)),
                ])
                pipe.fit(X_train[tr_idx], y_train[tr_idx])

                # predicciones OOF (clase)
                oof_pred[val_idx] = pipe.predict(X_train[val_idx])

                # scores OOF (por clase) si se puede
                fold_scores = get_scores_multiclass(pipe, X_train[val_idx])
                if fold_scores is not None:
                    if oof_scores is None:
                        oof_scores = np.full(
                            (len(y_train), fold_scores.shape[1]),
                            np.nan,
                            dtype=float
                        )
                    # asignar scores del fold a sus posiciones val_idx
                    oof_scores[val_idx, :] = fold_scores

            auc_oof = auc_multiclass_ovr(y_train, oof_scores, average=average)
            cv_metrics_all.append(compute_metrics(y_train, oof_pred, auc_value=auc_oof))

            # ---------- TEST ----------
            pipe_full = Pipeline([
                ("scaler", StandardScaler()),
                ("model", clone(estimator)),
            ])
            pipe_full.fit(X_train, y_train)

            test_pred = pipe_full.predict(X_test)

            test_scores = get_scores_multiclass(pipe_full, X_test)
            auc_test = auc_multiclass_ovr(y_test, test_scores, average=average)

            test_metrics_all.append(compute_metrics(y_test, test_pred, auc_value=auc_test))

        test_summary_rows.append(
            pd.Series(summarize(test_metrics_all, "Testing"), name=model_name)
        )
        cv_summary_rows.append(
            pd.Series(summarize(cv_metrics_all, "CV"), name=model_name)
        )

    df_test_summary = pd.DataFrame(test_summary_rows)[
        ["Accuracy_Testing", "Precision_Testing", "Recall_Testing", "F1_Testing", "AUC_Testing"]
    ]
    df_cv_summary = pd.DataFrame(cv_summary_rows)[
        ["Accuracy_CV", "Precision_CV", "Recall_CV", "F1_CV", "AUC_CV"]
    ]

    df_final_summary = pd.concat([df_test_summary, df_cv_summary], axis=1)
    return df_final_summary

# ALL SET OF FEATURES SC M AND NM

## HY3

In [4]:
X_HY3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY3_full.csv", index_col=0)
y_HY3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_full.csv", index_col=0)
print("X_HY3_data shape:", X_HY3_data.shape)
print("y_HY3_data shape:", y_HY3_data.shape)

X_HY3_data shape: (909, 931)
y_HY3_data shape: (909, 1)


In [5]:
df_HY3_full = evaluate_models_10x10_oof_and_test(
    X_df=X_HY3_data, 
    y_df=y_HY3_data, 
    models=classification_models, 
    random_state=42
)

df_HY3_full.head(10)


Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...


/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [16:27:03] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Evaluating adaboost...
Evaluating logistic_regression...
Evaluating sgd...


/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid valu

Evaluating svc...
Evaluating knn...
Evaluating gaussian_nb...


,Accuracy_Testing,Precision_Testing,Recall_Testing,F1_Testing,AUC_Testing,Accuracy_CV,Precision_CV,Recall_CV,F1_CV,AUC_CV
decision_tree,0.8341 ± 0.0230,0.6250 ± 0.0609,0.6170 ± 0.0406,0.6172 ± 0.0448,0.7580 ± 0.0258,0.8253 ± 0.0097,0.6455 ± 0.0383,0.6448 ± 0.0312,0.6440 ± 0.0336,0.7685 ± 0.0168
random_forest,0.9005 ± 0.0131,0.6009 ± 0.0089,0.6183 ± 0.0091,0.6093 ± 0.0090,0.9435 ± 0.0181,0.8986 ± 0.0049,0.7164 ± 0.1118,0.6264 ± 0.0083,0.6262 ± 0.0152,0.9532 ± 0.0088
extra_trees,0.8995 ± 0.0132,0.6004 ± 0.0091,0.6173 ± 0.0093,0.6085 ± 0.0092,0.9460 ± 0.0226,0.8967 ± 0.0049,0.6155 ± 0.0530,0.6168 ± 0.0065,0.6099 ± 0.0104,0.9543 ± 0.0071
xgboost,0.9011 ± 0.0147,0.7770 ± 0.1724,0.6631 ± 0.0484,0.6795 ± 0.0714,0.9105 ± 0.0155,0.8963 ± 0.0040,0.8004 ± 0.0309,0.6833 ± 0.0254,0.7094 ± 0.0318,0.9425 ± 0.0060
adaboost,0.8681 ± 0.0294,0.6820 ± 0.1045,0.6584 ± 0.0520,0.6595 ± 0.0535,0.9065 ± 0.0152,0.8747 ± 0.0077,0.7405 ± 0.0317,0.7176 ± 0.0190,0.7260 ± 0.0194,0.9213 ± 0.0051
logistic_regression,0.8648 ± 0.0257,0.6978 ± 0.1387,0.6497 ± 0.0632,0.6591 ± 0.0782,0.9022 ± 0.0333,0.8711 ± 0.0100,0.7417 ± 0.0391,0.6826 ± 0.0232,0.7024 ± 0.0274,0.8988 ± 0.0174
sgd,0.8599 ± 0.0198,0.7090 ± 0.1573,0.6379 ± 0.0681,0.6502 ± 0.0871,0.8454 ± 0.0282,0.8535 ± 0.0106,0.7381 ± 0.0286,0.6432 ± 0.0265,0.6659 ± 0.0291,0.8425 ± 0.0227
svc,0.8940 ± 0.0206,0.5963 ± 0.0136,0.6136 ± 0.0144,0.6046 ± 0.0140,nan ± nan,0.8878 ± 0.0045,0.5919 ± 0.0030,0.6093 ± 0.0031,0.6004 ± 0.0030,nan ± nan
knn,0.7527 ± 0.0201,0.5269 ± 0.0119,0.5258 ± 0.0132,0.5052 ± 0.0147,0.7672 ± 0.0256,0.7512 ± 0.0046,0.5418 ± 0.0535,0.5267 ± 0.0055,0.5076 ± 0.0095,0.7856 ± 0.0119
gaussian_nb,0.6445 ± 0.0828,0.5991 ± 0.0167,0.6229 ± 0.0737,0.5178 ± 0.0561,0.7881 ± 0.0459,0.6682 ± 0.0863,0.5977 ± 0.0197,0.6087 ± 0.0548,0.5316 ± 0.0574,0.7748 ± 0.0293


## HY4

In [6]:
X_HY4_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)
y_HY4_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_full.csv", index_col=0)
print("X_HY4_data shape:", X_HY4_data.shape)
print("y_HY4_data shape:", y_HY4_data.shape)

X_HY4_data shape: (909, 931)
y_HY4_data shape: (909, 1)


In [7]:
df_HY4_full = evaluate_models_10x10_oof_and_test(
    X_df=X_HY4_data, 
    y_df=y_HY4_data, 
    models=classification_models, 
    random_state=42
)

df_HY4_full.head(10)

Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating logistic_regression...
Evaluating sgd...


/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid valu

Evaluating svc...
Evaluating knn...
Evaluating gaussian_nb...


,Accuracy_Testing,Precision_Testing,Recall_Testing,F1_Testing,AUC_Testing,Accuracy_CV,Precision_CV,Recall_CV,F1_CV,AUC_CV
decision_tree,0.7055 ± 0.0242,0.5088 ± 0.0280,0.5067 ± 0.0377,0.5032 ± 0.0304,0.7004 ± 0.0216,0.7142 ± 0.0147,0.5067 ± 0.0224,0.5131 ± 0.0254,0.5090 ± 0.0238,0.7042 ± 0.0142
random_forest,0.8335 ± 0.0132,0.5307 ± 0.1244,0.4913 ± 0.0131,0.4648 ± 0.0231,0.8911 ± 0.0151,0.8304 ± 0.0047,0.5540 ± 0.1243,0.4835 ± 0.0056,0.4528 ± 0.0096,0.8854 ± 0.0081
extra_trees,0.8291 ± 0.0148,0.4785 ± 0.1064,0.4832 ± 0.0099,0.4519 ± 0.0136,0.8877 ± 0.0206,0.8292 ± 0.0052,0.5658 ± 0.1423,0.4833 ± 0.0070,0.4528 ± 0.0118,0.8825 ± 0.0046
xgboost,0.8308 ± 0.0173,0.6931 ± 0.1337,0.5456 ± 0.0409,0.5535 ± 0.0559,0.8854 ± 0.0270,0.8228 ± 0.0058,0.6521 ± 0.0580,0.5256 ± 0.0251,0.5299 ± 0.0354,0.8657 ± 0.0132
adaboost,0.7775 ± 0.0297,0.6311 ± 0.0813,0.5441 ± 0.0385,0.5568 ± 0.0394,0.8559 ± 0.0240,0.7744 ± 0.0097,0.5855 ± 0.0282,0.5494 ± 0.0264,0.5606 ± 0.0270,0.8499 ± 0.0087
logistic_regression,0.7571 ± 0.0305,0.5503 ± 0.0758,0.5376 ± 0.0634,0.5367 ± 0.0650,0.8087 ± 0.0331,0.7565 ± 0.0120,0.5503 ± 0.0368,0.5273 ± 0.0306,0.5353 ± 0.0324,0.8030 ± 0.0145
sgd,0.7753 ± 0.0161,0.6143 ± 0.1023,0.5450 ± 0.0499,0.5554 ± 0.0558,0.8116 ± 0.0224,0.7730 ± 0.0077,0.5670 ± 0.0370,0.5240 ± 0.0188,0.5331 ± 0.0243,nan ± nan
svc,0.8187 ± 0.0119,0.4105 ± 0.0062,0.4745 ± 0.0070,0.4394 ± 0.0064,nan ± nan,0.8220 ± 0.0055,0.4115 ± 0.0027,0.4757 ± 0.0032,0.4409 ± 0.0029,nan ± nan
knn,0.6615 ± 0.0225,0.3760 ± 0.0331,0.3827 ± 0.0176,0.3582 ± 0.0231,0.7219 ± 0.0293,0.6644 ± 0.0074,0.4232 ± 0.0390,0.3911 ± 0.0064,0.3733 ± 0.0098,0.7128 ± 0.0076
gaussian_nb,0.4654 ± 0.0511,0.4394 ± 0.0332,0.4808 ± 0.0751,0.3609 ± 0.0409,0.7269 ± 0.0464,0.4717 ± 0.0380,0.4514 ± 0.0164,0.4663 ± 0.0341,0.3668 ± 0.0255,0.7205 ± 0.0257


## MCID

In [8]:
X_MCID_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_full.csv", index_col=0)
y_MCID_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_full.csv", index_col=0)
print("X_MCID_data shape:", X_MCID_data.shape)
print("y_MCID_data shape:", y_MCID_data.shape)

X_MCID_data shape: (909, 936)
y_MCID_data shape: (909, 1)


In [9]:
df_MCID_full = evaluate_models_10x10_oof_and_test(
    X_df=X_MCID_data, 
    y_df=y_MCID_data, 
    models=classification_models, 
    random_state=42
)

df_MCID_full.head(10)

Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating logistic_regression...
Evaluating sgd...


/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid valu

Evaluating svc...
Evaluating knn...
Evaluating gaussian_nb...


,Accuracy_Testing,Precision_Testing,Recall_Testing,F1_Testing,AUC_Testing,Accuracy_CV,Precision_CV,Recall_CV,F1_CV,AUC_CV
decision_tree,0.4841 ± 0.0346,0.4011 ± 0.0363,0.4034 ± 0.0358,0.4013 ± 0.0360,0.5629 ± 0.0275,0.4784 ± 0.0160,0.4002 ± 0.0168,0.3999 ± 0.0170,0.3998 ± 0.0169,0.5582 ± 0.0130
random_forest,0.5819 ± 0.0344,0.4621 ± 0.0366,0.4566 ± 0.0289,0.4311 ± 0.0271,0.7143 ± 0.0304,0.5773 ± 0.0117,0.4622 ± 0.0202,0.4517 ± 0.0132,0.4285 ± 0.0136,0.7070 ± 0.0111
extra_trees,0.5775 ± 0.0208,0.4678 ± 0.0444,0.4517 ± 0.0229,0.4289 ± 0.0209,0.7124 ± 0.0232,0.5795 ± 0.0125,0.4373 ± 0.0227,0.4528 ± 0.0140,0.4256 ± 0.0142,0.7056 ± 0.0096
xgboost,0.5621 ± 0.0350,0.4445 ± 0.0515,0.4439 ± 0.0368,0.4337 ± 0.0402,0.6939 ± 0.0277,0.5594 ± 0.0131,0.4448 ± 0.0276,0.4421 ± 0.0178,0.4343 ± 0.0208,0.6912 ± 0.0137
adaboost,0.5604 ± 0.0230,0.4588 ± 0.0473,0.4517 ± 0.0313,0.4403 ± 0.0309,0.6823 ± 0.0290,0.5510 ± 0.0087,0.4350 ± 0.0178,0.4366 ± 0.0110,0.4260 ± 0.0128,0.6652 ± 0.0105
logistic_regression,0.4995 ± 0.0330,0.4242 ± 0.0299,0.4234 ± 0.0308,0.4226 ± 0.0307,0.6039 ± 0.0381,0.4988 ± 0.0187,0.4211 ± 0.0215,0.4208 ± 0.0219,0.4208 ± 0.0217,0.6063 ± 0.0215
sgd,0.5187 ± 0.0372,0.4235 ± 0.0352,0.4180 ± 0.0321,0.4165 ± 0.0327,0.5982 ± 0.0291,0.5139 ± 0.0157,0.4122 ± 0.0162,0.4104 ± 0.0148,0.4085 ± 0.0151,0.5960 ± 0.0132
svc,0.5945 ± 0.0387,0.4433 ± 0.1372,0.4559 ± 0.0296,0.4142 ± 0.0278,nan ± nan,0.5836 ± 0.0134,0.4060 ± 0.0574,0.4445 ± 0.0193,0.4046 ± 0.0204,nan ± nan
knn,0.5604 ± 0.0206,0.4545 ± 0.0294,0.4150 ± 0.0123,0.3985 ± 0.0152,0.6438 ± 0.0289,0.5549 ± 0.0112,0.4547 ± 0.0283,0.4111 ± 0.0134,0.3967 ± 0.0179,0.6396 ± 0.0139
gaussian_nb,0.4527 ± 0.0786,0.4664 ± 0.0441,0.4425 ± 0.0514,0.4031 ± 0.0676,0.6512 ± 0.0361,0.4503 ± 0.0487,0.4787 ± 0.0129,0.4484 ± 0.0184,0.4084 ± 0.0331,0.6527 ± 0.0121


# SET OF FEATURES M

## HY3

In [10]:
cols=datos['M_data']
remove_cols = ['NHY_mean', 'NHY_min', 'NHY_max', 'NHY_var', 'NHY_std']
valid_cols = [x for x in cols if x not in remove_cols]
X_HY3_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY3_full.csv", index_col=0)[valid_cols]
y_HY3_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_full.csv", index_col=0)
print("X_HY3_motor_data shape:", X_HY3_motor_data.shape)
print("y_HY3_motor_data shape:", y_HY3_motor_data.shape)

X_HY3_motor_data shape: (909, 300)
y_HY3_motor_data shape: (909, 1)


In [11]:
df_HY3_motor_data = evaluate_models_10x10_oof_and_test(
    X_df=X_HY3_motor_data, 
    y_df=y_HY3_motor_data, 
    models=classification_models, 
    random_state=42
)

df_HY3_motor_data.head(10)

Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating logistic_regression...


/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Evaluating sgd...
Evaluating svc...
Evaluating knn...
Evaluating gaussian_nb...


,Accuracy_Testing,Precision_Testing,Recall_Testing,F1_Testing,AUC_Testing,Accuracy_CV,Precision_CV,Recall_CV,F1_CV,AUC_CV
decision_tree,0.8473 ± 0.0224,0.6320 ± 0.0590,0.6383 ± 0.0614,0.6334 ± 0.0582,0.7724 ± 0.0350,0.8341 ± 0.0131,0.6644 ± 0.0313,0.6664 ± 0.0255,0.6647 ± 0.0283,0.7818 ± 0.0154
random_forest,0.9055 ± 0.0096,0.7042 ± 0.1601,0.6411 ± 0.0309,0.6455 ± 0.0525,0.9455 ± 0.0200,0.9011 ± 0.0047,0.8026 ± 0.0686,0.6519 ± 0.0172,0.6680 ± 0.0273,0.9546 ± 0.0051
extra_trees,0.9011 ± 0.0116,0.7019 ± 0.1617,0.6375 ± 0.0334,0.6427 ± 0.0545,0.9436 ± 0.0149,0.8961 ± 0.0047,0.7838 ± 0.1095,0.6450 ± 0.0182,0.6595 ± 0.0312,0.9571 ± 0.0080
xgboost,0.8984 ± 0.0172,0.7750 ± 0.1720,0.6610 ± 0.0478,0.6777 ± 0.0709,0.8975 ± 0.0246,0.8905 ± 0.0058,0.7798 ± 0.0364,0.6901 ± 0.0271,0.7150 ± 0.0313,0.9409 ± 0.0053
adaboost,0.8692 ± 0.0364,0.6867 ± 0.1119,0.6532 ± 0.0624,0.6590 ± 0.0661,0.9096 ± 0.0182,0.8737 ± 0.0100,0.7346 ± 0.0459,0.7266 ± 0.0323,0.7291 ± 0.0364,0.9259 ± 0.0079
logistic_regression,0.8484 ± 0.0298,0.6684 ± 0.0719,0.6518 ± 0.0469,0.6564 ± 0.0565,0.8674 ± 0.0274,0.8583 ± 0.0136,0.7114 ± 0.0393,0.6975 ± 0.0291,0.7029 ± 0.0318,0.9053 ± 0.0168
sgd,0.8527 ± 0.0230,0.6801 ± 0.0804,0.6411 ± 0.0393,0.6514 ± 0.0519,0.8590 ± 0.0252,0.8536 ± 0.0116,0.7304 ± 0.0367,0.6912 ± 0.0232,0.7062 ± 0.0260,0.8846 ± 0.0108
svc,0.8995 ± 0.0217,0.5997 ± 0.0145,0.6177 ± 0.0149,0.6083 ± 0.0147,nan ± nan,0.8974 ± 0.0057,0.5981 ± 0.0038,0.6164 ± 0.0040,0.6069 ± 0.0039,nan ± nan
knn,0.8505 ± 0.0257,0.6066 ± 0.1101,0.5946 ± 0.0296,0.5856 ± 0.0434,0.8360 ± 0.0285,0.8558 ± 0.0053,0.7627 ± 0.0743,0.6235 ± 0.0202,0.6334 ± 0.0325,0.8621 ± 0.0129
gaussian_nb,0.8038 ± 0.0242,0.6505 ± 0.0236,0.7582 ± 0.0789,0.6444 ± 0.0345,0.8806 ± 0.0699,0.8139 ± 0.0123,0.6643 ± 0.0098,0.8185 ± 0.0229,0.6685 ± 0.0146,0.9005 ± 0.0175


## HY4

In [12]:
cols=datos['M_data']
remove_cols = ['NHY_mean', 'NHY_min', 'NHY_max', 'NHY_var', 'NHY_std']
valid_cols = [x for x in cols if x not in remove_cols]
X_HY4_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)[valid_cols]
y_HY4_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_full.csv", index_col=0)
print("X_HY4_motor_data shape:", X_HY4_motor_data.shape)
print("y_HY4_motor_data shape:", y_HY4_motor_data.shape)

X_HY4_motor_data shape: (909, 300)
y_HY4_motor_data shape: (909, 1)


In [13]:
df_HY4_motor_data = evaluate_models_10x10_oof_and_test(
    X_df=X_HY4_motor_data, 
    y_df=y_HY4_motor_data, 
    models=classification_models, 
    random_state=42
)

df_HY4_motor_data.head(10)

Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating logistic_regression...


/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/st

Evaluating sgd...
Evaluating svc...
Evaluating knn...
Evaluating gaussian_nb...


,Accuracy_Testing,Precision_Testing,Recall_Testing,F1_Testing,AUC_Testing,Accuracy_CV,Precision_CV,Recall_CV,F1_CV,AUC_CV
decision_tree,0.7148 ± 0.0290,0.5152 ± 0.0656,0.5225 ± 0.0633,0.5122 ± 0.0574,0.7103 ± 0.0348,0.7227 ± 0.0092,0.5258 ± 0.0277,0.5285 ± 0.0311,0.5259 ± 0.0284,0.7139 ± 0.0172
random_forest,0.8418 ± 0.0155,0.6993 ± 0.1199,0.5325 ± 0.0424,0.5350 ± 0.0610,0.9029 ± 0.0150,0.8362 ± 0.0043,0.7334 ± 0.0649,0.5215 ± 0.0138,0.5218 ± 0.0232,0.8939 ± 0.0047
extra_trees,0.8352 ± 0.0119,0.6611 ± 0.1125,0.5082 ± 0.0195,0.4988 ± 0.0319,0.9026 ± 0.0121,0.8351 ± 0.0060,0.7528 ± 0.0461,0.5191 ± 0.0095,0.5179 ± 0.0157,0.8933 ± 0.0041
xgboost,0.8154 ± 0.0149,0.6051 ± 0.0881,0.5414 ± 0.0338,0.5488 ± 0.0486,0.8771 ± 0.0300,0.8136 ± 0.0074,0.6288 ± 0.0427,0.5444 ± 0.0199,0.5569 ± 0.0274,0.8633 ± 0.0147
adaboost,0.7907 ± 0.0191,0.6066 ± 0.0750,0.5770 ± 0.0402,0.5811 ± 0.0478,0.8623 ± 0.0158,0.7733 ± 0.0131,0.5792 ± 0.0351,0.5485 ± 0.0212,0.5585 ± 0.0249,0.8498 ± 0.0097
logistic_regression,0.7346 ± 0.0269,0.5278 ± 0.0327,0.5280 ± 0.0290,0.5256 ± 0.0296,0.8073 ± 0.0309,0.7303 ± 0.0119,0.5277 ± 0.0167,0.5204 ± 0.0189,0.5234 ± 0.0172,0.7994 ± 0.0149
sgd,0.7357 ± 0.0287,0.5486 ± 0.0546,0.5357 ± 0.0355,0.5384 ± 0.0377,0.7981 ± 0.0217,0.7318 ± 0.0092,0.5409 ± 0.0336,0.5201 ± 0.0274,0.5282 ± 0.0299,0.7803 ± 0.0095
svc,0.8247 ± 0.0196,0.5145 ± 0.1194,0.4827 ± 0.0146,0.4543 ± 0.0193,nan ± nan,0.8243 ± 0.0048,0.5299 ± 0.1144,0.4794 ± 0.0031,0.4482 ± 0.0056,nan ± nan
knn,0.7632 ± 0.0213,0.5623 ± 0.1315,0.4786 ± 0.0312,0.4809 ± 0.0476,0.7955 ± 0.0358,0.7631 ± 0.0073,0.5934 ± 0.1028,0.4799 ± 0.0196,0.4849 ± 0.0326,0.7897 ± 0.0094
gaussian_nb,0.6137 ± 0.0358,0.5220 ± 0.0181,0.6285 ± 0.0331,0.4846 ± 0.0314,0.8433 ± 0.0219,0.6147 ± 0.0242,0.5154 ± 0.0087,0.6094 ± 0.0181,0.4783 ± 0.0203,0.8314 ± 0.0051


## MCID

In [14]:
cols=datos['M_data']
X_MCID_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_full.csv", index_col=0)[cols]
y_MCID_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_full.csv", index_col=0)
print("X_MCID_motor_data shape:", X_MCID_motor_data.shape)
print("y_MCID_motor_data shape:", y_MCID_motor_data.shape)

X_MCID_motor_data shape: (909, 305)
y_MCID_motor_data shape: (909, 1)


In [15]:
df_MCID_motor_data = evaluate_models_10x10_oof_and_test(
    X_df=X_MCID_motor_data, 
    y_df=y_MCID_motor_data, 
    models=classification_models, 
    random_state=42
)

df_MCID_motor_data.head(10)

Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating logistic_regression...


/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/st

Evaluating sgd...
Evaluating svc...
Evaluating knn...
Evaluating gaussian_nb...


,Accuracy_Testing,Precision_Testing,Recall_Testing,F1_Testing,AUC_Testing,Accuracy_CV,Precision_CV,Recall_CV,F1_CV,AUC_CV
decision_tree,0.4918 ± 0.0301,0.4204 ± 0.0412,0.4190 ± 0.0426,0.4170 ± 0.0414,0.5725 ± 0.0296,0.4812 ± 0.0119,0.4055 ± 0.0115,0.4061 ± 0.0116,0.4054 ± 0.0115,0.5638 ± 0.0091
random_forest,0.5775 ± 0.0292,0.4625 ± 0.0675,0.4554 ± 0.0323,0.4313 ± 0.0296,0.7027 ± 0.0212,0.5722 ± 0.0132,0.4562 ± 0.0302,0.4532 ± 0.0152,0.4316 ± 0.0185,0.6950 ± 0.0069
extra_trees,0.5725 ± 0.0228,0.4566 ± 0.0498,0.4536 ± 0.0237,0.4312 ± 0.0262,0.6997 ± 0.0182,0.5611 ± 0.0111,0.4413 ± 0.0243,0.4458 ± 0.0131,0.4272 ± 0.0152,0.6884 ± 0.0107
xgboost,0.5418 ± 0.0321,0.4286 ± 0.0533,0.4301 ± 0.0373,0.4241 ± 0.0426,0.6716 ± 0.0265,0.5338 ± 0.0086,0.4211 ± 0.0139,0.4245 ± 0.0104,0.4184 ± 0.0115,0.6729 ± 0.0084
adaboost,0.5571 ± 0.0253,0.4566 ± 0.0456,0.4484 ± 0.0305,0.4389 ± 0.0295,0.6950 ± 0.0283,0.5538 ± 0.0090,0.4463 ± 0.0159,0.4454 ± 0.0095,0.4372 ± 0.0122,0.6786 ± 0.0121
logistic_regression,0.5132 ± 0.0286,0.4207 ± 0.0276,0.4200 ± 0.0250,0.4187 ± 0.0262,0.5708 ± 0.0398,0.5061 ± 0.0185,0.4188 ± 0.0205,0.4173 ± 0.0198,0.4175 ± 0.0202,0.5741 ± 0.0257
sgd,0.4885 ± 0.0453,0.4069 ± 0.0445,0.4061 ± 0.0432,0.4043 ± 0.0430,0.5474 ± 0.0460,0.4882 ± 0.0188,0.4051 ± 0.0222,0.4051 ± 0.0217,0.4049 ± 0.0219,0.5648 ± 0.0245
svc,0.5852 ± 0.0307,0.3840 ± 0.0522,0.4537 ± 0.0266,0.4100 ± 0.0274,nan ± nan,0.5754 ± 0.0140,0.4047 ± 0.0358,0.4491 ± 0.0136,0.4067 ± 0.0118,nan ± nan
knn,0.5456 ± 0.0312,0.4401 ± 0.0518,0.4212 ± 0.0316,0.4109 ± 0.0367,0.6644 ± 0.0205,0.5451 ± 0.0130,0.4363 ± 0.0199,0.4171 ± 0.0163,0.4093 ± 0.0189,0.6516 ± 0.0085
gaussian_nb,0.5418 ± 0.0436,0.4774 ± 0.0469,0.4717 ± 0.0384,0.4580 ± 0.0379,0.6902 ± 0.0307,0.5409 ± 0.0131,0.4788 ± 0.0160,0.4744 ± 0.0155,0.4636 ± 0.0161,0.6905 ± 0.0083


# SET OF FEATURES NM

## HY3

In [16]:
cols=datos['NM_data']
X_HY3_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY3_full.csv", index_col=0)[cols]
y_HY3_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_full.csv", index_col=0)
print("X_HY3_NO_motor_data shape:", X_HY3_NO_motor_data.shape)
print("y_HY3_NO_motor_data shape:", y_HY3_NO_motor_data.shape)

X_HY3_NO_motor_data shape: (909, 625)
y_HY3_NO_motor_data shape: (909, 1)


In [17]:
df_HY3_NO_motor_data = evaluate_models_10x10_oof_and_test(
    X_df=X_HY3_NO_motor_data, 
    y_df=y_HY3_NO_motor_data, 
    models=classification_models, 
    random_state=42
)

df_HY3_NO_motor_data.head(10)

Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating logistic_regression...


/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/st

Evaluating sgd...
Evaluating svc...
Evaluating knn...
Evaluating gaussian_nb...


,Accuracy_Testing,Precision_Testing,Recall_Testing,F1_Testing,AUC_Testing,Accuracy_CV,Precision_CV,Recall_CV,F1_CV,AUC_CV
decision_tree,0.5978 ± 0.0401,0.4194 ± 0.0314,0.4219 ± 0.0338,0.4193 ± 0.0324,0.5837 ± 0.0274,0.5618 ± 0.0180,0.4023 ± 0.0215,0.4032 ± 0.0236,0.4024 ± 0.0222,0.5631 ± 0.0157
random_forest,0.6813 ± 0.0421,0.4569 ± 0.0289,0.4636 ± 0.0285,0.4575 ± 0.0286,0.7284 ± 0.0391,0.6598 ± 0.0119,0.4407 ± 0.0081,0.4486 ± 0.0084,0.4430 ± 0.0084,0.7405 ± 0.0160
extra_trees,0.6676 ± 0.0313,0.4474 ± 0.0219,0.4541 ± 0.0212,0.4482 ± 0.0213,0.7400 ± 0.0484,0.6589 ± 0.0115,0.4413 ± 0.0079,0.4467 ± 0.0081,0.4411 ± 0.0081,0.7425 ± 0.0227
xgboost,0.6769 ± 0.0312,0.4525 ± 0.0211,0.4623 ± 0.0211,0.4559 ± 0.0211,0.7385 ± 0.0345,0.6718 ± 0.0149,0.4814 ± 0.1023,0.4603 ± 0.0092,0.4561 ± 0.0110,0.7480 ± 0.0185
adaboost,0.6412 ± 0.0328,0.4432 ± 0.0278,0.4463 ± 0.0196,0.4411 ± 0.0215,0.6816 ± 0.0262,0.6281 ± 0.0180,0.4467 ± 0.0417,0.4368 ± 0.0213,0.4371 ± 0.0262,0.6913 ± 0.0155
logistic_regression,0.6297 ± 0.0245,0.5000 ± 0.1106,0.4742 ± 0.0705,0.4768 ± 0.0735,0.7031 ± 0.0387,0.6109 ± 0.0117,0.4674 ± 0.0315,0.4461 ± 0.0146,0.4521 ± 0.0192,0.6721 ± 0.0256
sgd,0.6302 ± 0.0324,0.5052 ± 0.1265,0.4560 ± 0.0428,0.4638 ± 0.0587,0.6542 ± 0.0576,0.6062 ± 0.0086,0.4614 ± 0.0587,0.4284 ± 0.0155,0.4328 ± 0.0231,0.6269 ± 0.0128
svc,0.6797 ± 0.0226,0.4552 ± 0.0151,0.4630 ± 0.0161,0.4568 ± 0.0161,nan ± nan,0.6591 ± 0.0111,0.4405 ± 0.0075,0.4479 ± 0.0079,0.4423 ± 0.0079,nan ± nan
knn,0.5599 ± 0.0361,0.3804 ± 0.0239,0.3904 ± 0.0235,0.3755 ± 0.0264,0.5784 ± 0.0306,0.5451 ± 0.0115,0.3713 ± 0.0080,0.3807 ± 0.0080,0.3661 ± 0.0080,0.5672 ± 0.0102
gaussian_nb,0.2665 ± 0.1442,0.3865 ± 0.0505,0.3799 ± 0.1162,0.2396 ± 0.1049,0.5391 ± 0.0660,0.3037 ± 0.1159,0.3875 ± 0.0189,0.3753 ± 0.0842,0.2685 ± 0.0767,0.5371 ± 0.0475


## HY4

In [18]:
cols=datos['NM_data']
X_HY4_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)[cols]
y_HY4_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_full.csv", index_col=0)
print("X_HY4_NO_motor_data shape:", X_HY4_NO_motor_data.shape)
print("y_HY4_NO_motor_data shape:", y_HY4_NO_motor_data.shape)

X_HY4_NO_motor_data shape: (909, 625)
y_HY4_NO_motor_data shape: (909, 1)


In [19]:
df_HY4_NO_motor_data = evaluate_models_10x10_oof_and_test(
    X_df=X_HY4_NO_motor_data, 
    y_df=y_HY4_NO_motor_data, 
    models=classification_models, 
    random_state=42
)

df_HY4_NO_motor_data.head(10)

Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating logistic_regression...


/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/st

Evaluating sgd...
Evaluating svc...
Evaluating knn...
Evaluating gaussian_nb...


,Accuracy_Testing,Precision_Testing,Recall_Testing,F1_Testing,AUC_Testing,Accuracy_CV,Precision_CV,Recall_CV,F1_CV,AUC_CV
decision_tree,0.4676 ± 0.0453,0.3073 ± 0.0496,0.3025 ± 0.0374,0.3026 ± 0.0388,0.5447 ± 0.0256,0.4492 ± 0.0129,0.2982 ± 0.0100,0.2985 ± 0.0091,0.2980 ± 0.0096,0.5392 ± 0.0056
random_forest,0.5978 ± 0.0357,0.2985 ± 0.0183,0.3458 ± 0.0210,0.3200 ± 0.0195,0.6905 ± 0.0328,0.6062 ± 0.0155,0.3025 ± 0.0079,0.3502 ± 0.0091,0.3246 ± 0.0084,0.6809 ± 0.0139
extra_trees,0.5995 ± 0.0338,0.3000 ± 0.0170,0.3472 ± 0.0197,0.3214 ± 0.0182,0.6962 ± 0.0287,0.6069 ± 0.0160,0.3029 ± 0.0080,0.3506 ± 0.0093,0.3250 ± 0.0086,0.6796 ± 0.0128
xgboost,0.6044 ± 0.0235,0.3329 ± 0.0449,0.3535 ± 0.0139,0.3339 ± 0.0162,0.6882 ± 0.0229,0.5950 ± 0.0091,0.3487 ± 0.0538,0.3487 ± 0.0053,0.3315 ± 0.0085,0.6654 ± 0.0189
adaboost,0.5648 ± 0.0339,0.3022 ± 0.0488,0.3285 ± 0.0187,0.3079 ± 0.0187,0.6750 ± 0.0249,0.5655 ± 0.0148,0.3296 ± 0.0359,0.3321 ± 0.0072,0.3165 ± 0.0067,0.6492 ± 0.0249
logistic_regression,0.5033 ± 0.0332,0.3544 ± 0.0617,0.3504 ± 0.0504,0.3480 ± 0.0508,0.6194 ± 0.0260,0.5102 ± 0.0187,0.3564 ± 0.0260,0.3417 ± 0.0232,0.3464 ± 0.0239,0.6192 ± 0.0195
sgd,0.4995 ± 0.0318,0.3122 ± 0.0356,0.3156 ± 0.0303,0.3123 ± 0.0326,0.6011 ± 0.0267,0.5232 ± 0.0148,0.3428 ± 0.0320,0.3289 ± 0.0170,0.3296 ± 0.0208,0.6006 ± 0.0146
svc,0.5918 ± 0.0288,0.2954 ± 0.0148,0.3423 ± 0.0169,0.3169 ± 0.0157,nan ± nan,0.5990 ± 0.0130,0.2991 ± 0.0064,0.3462 ± 0.0074,0.3209 ± 0.0068,nan ± nan
knn,0.4907 ± 0.0299,0.3049 ± 0.0470,0.2902 ± 0.0182,0.2663 ± 0.0242,0.5573 ± 0.0198,0.4854 ± 0.0112,0.2875 ± 0.0147,0.2847 ± 0.0089,0.2589 ± 0.0112,0.5654 ± 0.0116
gaussian_nb,0.1110 ± 0.0307,0.2700 ± 0.0515,0.2477 ± 0.0757,0.1088 ± 0.0271,0.5272 ± 0.0500,0.1241 ± 0.0182,0.2618 ± 0.0226,0.2768 ± 0.0454,0.1181 ± 0.0130,0.5288 ± 0.0262


## MCID

In [20]:
cols=datos['NM_data']
X_MCID_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_full.csv", index_col=0)[cols]
y_MCID_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_full.csv", index_col=0)
print("X_MCID_NO_motor_data shape:", X_MCID_NO_motor_data.shape)
print("y_MCID_NO_motor_data shape:", y_MCID_NO_motor_data.shape)

X_MCID_NO_motor_data shape: (909, 625)
y_MCID_NO_motor_data shape: (909, 1)


In [21]:
df_MCID_NO_motor_data = evaluate_models_10x10_oof_and_test(
    X_df=X_MCID_NO_motor_data, 
    y_df=y_MCID_NO_motor_data, 
    models=classification_models, 
    random_state=42
)

df_MCID_NO_motor_data.head(10)

Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating logistic_regression...


/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/st

Evaluating sgd...
Evaluating svc...
Evaluating knn...
Evaluating gaussian_nb...


,Accuracy_Testing,Precision_Testing,Recall_Testing,F1_Testing,AUC_Testing,Accuracy_CV,Precision_CV,Recall_CV,F1_CV,AUC_CV
decision_tree,0.4319 ± 0.0204,0.3705 ± 0.0184,0.3697 ± 0.0198,0.3694 ± 0.0195,0.5294 ± 0.0149,0.4231 ± 0.0132,0.3592 ± 0.0119,0.3592 ± 0.0125,0.3589 ± 0.0122,0.5218 ± 0.0098
random_forest,0.5319 ± 0.0224,0.4015 ± 0.1549,0.3581 ± 0.0200,0.3073 ± 0.0254,0.5805 ± 0.0322,0.5326 ± 0.0081,0.3493 ± 0.0487,0.3609 ± 0.0109,0.3136 ± 0.0158,0.5832 ± 0.0104
extra_trees,0.5346 ± 0.0209,0.3429 ± 0.0694,0.3610 ± 0.0169,0.3102 ± 0.0193,0.5801 ± 0.0283,0.5333 ± 0.0102,0.3708 ± 0.0601,0.3617 ± 0.0113,0.3151 ± 0.0151,0.5801 ± 0.0172
xgboost,0.5066 ± 0.0330,0.3787 ± 0.0511,0.3655 ± 0.0289,0.3442 ± 0.0327,0.5676 ± 0.0270,0.5069 ± 0.0149,0.3719 ± 0.0238,0.3700 ± 0.0155,0.3521 ± 0.0183,0.5707 ± 0.0178
adaboost,0.5247 ± 0.0278,0.3388 ± 0.0723,0.3620 ± 0.0204,0.3184 ± 0.0227,0.5552 ± 0.0324,0.5191 ± 0.0127,0.3576 ± 0.0430,0.3657 ± 0.0172,0.3325 ± 0.0225,0.5525 ± 0.0222
logistic_regression,0.4456 ± 0.0402,0.3881 ± 0.0399,0.3877 ± 0.0414,0.3861 ± 0.0409,0.5630 ± 0.0386,0.4385 ± 0.0206,0.3756 ± 0.0170,0.3758 ± 0.0176,0.3751 ± 0.0175,0.5483 ± 0.0162
sgd,0.4527 ± 0.0246,0.3887 ± 0.0225,0.3858 ± 0.0220,0.3862 ± 0.0223,0.5554 ± 0.0371,0.4488 ± 0.0182,0.3752 ± 0.0157,0.3751 ± 0.0155,0.3748 ± 0.0158,0.5439 ± 0.0141
svc,0.5346 ± 0.0097,0.3284 ± 0.0290,0.3496 ± 0.0092,0.2809 ± 0.0188,nan ± nan,0.5399 ± 0.0075,0.3189 ± 0.0232,0.3507 ± 0.0110,0.2827 ± 0.0212,nan ± nan
knn,0.5126 ± 0.0339,0.3884 ± 0.0888,0.3633 ± 0.0365,0.3354 ± 0.0488,0.5742 ± 0.0218,0.5107 ± 0.0097,0.3879 ± 0.0154,0.3654 ± 0.0091,0.3442 ± 0.0108,0.5666 ± 0.0117
gaussian_nb,0.2505 ± 0.0583,0.4407 ± 0.0806,0.3483 ± 0.0298,0.2266 ± 0.0712,0.5507 ± 0.0300,0.2593 ± 0.0407,0.4141 ± 0.0337,0.3518 ± 0.0121,0.2454 ± 0.0498,0.5499 ± 0.0105


# AGNOSTIC FEATURE SELECTION FULL 
- Variance treshold
- Correlation de spearman
- MI
- Chi2

In [22]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import mutual_info_classif, chi2
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)


# =========================================================
# Feature selectors
# =========================================================

class SpearmanCorrelationDiscard(BaseEstimator, TransformerMixin):
    """
    Elimina variables altamente correlacionadas entre sí
    usando Spearman. De cada par altamente correlacionado,
    elimina la que tenga menor correlación absoluta con y.
    """

    def __init__(self, threshold=0.9):
        self.threshold = threshold
        self.vars_to_drop_ = None
        self.feature_names_in_ = None

    def fit(self, X, y=None):
        X = X.copy()

        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        self.feature_names_in_ = X.columns.to_list()

        data = X.copy()
        data["_target_"] = y

        corr_df = data.corr(method="spearman")

        # máscara triangular inferior, sin diagonal
        mask = np.tril(np.ones(corr_df.shape), k=-1).astype(bool)

        corr_long = (
            corr_df.where(mask)
            .stack()
            .reset_index()
            .rename(columns={"level_0": "V1", "level_1": "V2", 0: "CORR"})
        )

        # quitar pares donde aparezca el target
        corr_long = corr_long[
            (corr_long["V1"] != "_target_") & (corr_long["V2"] != "_target_")
        ].copy()

        target_corr = corr_df["_target_"]

        corr_long["V1target"] = corr_long["V1"].map(target_corr)
        corr_long["V2target"] = corr_long["V2"].map(target_corr)

        corr_long["WORST_VAR"] = np.where(
            abs(corr_long["V1target"]) <= abs(corr_long["V2target"]),
            corr_long["V1"],
            corr_long["V2"]
        )

        discard_corr_long = corr_long.loc[corr_long["CORR"].abs() > self.threshold]
        self.vars_to_drop_ = list(set(discard_corr_long["WORST_VAR"]))

        return self

    def transform(self, X):
        X = X.copy()

        if isinstance(X, pd.DataFrame):
            return X.drop(columns=self.vars_to_drop_, errors="ignore")

        X_df = pd.DataFrame(X, columns=self.feature_names_in_)
        X_df = X_df.drop(columns=self.vars_to_drop_, errors="ignore")
        return X_df


class MIThresholdSelector(BaseEstimator, TransformerMixin):
    """
    Selecciona variables con mutual_info_classif >= threshold.
    """

    def __init__(self, threshold=0.01, random_state=42):
        self.threshold = threshold
        self.random_state = random_state
        self.support_ = None
        self.mi_scores_ = None
        self.feature_names_in_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.to_list()
            X_fit = X.values
        else:
            self.feature_names_in_ = None
            X_fit = X

        self.mi_scores_ = mutual_info_classif(X_fit, y, random_state=self.random_state)
        self.support_ = self.mi_scores_ >= self.threshold

        # si no sobrevive ninguna, dejar la mejor
        if not np.any(self.support_):
            best_idx = np.argmax(self.mi_scores_)
            self.support_[best_idx] = True

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            cols = X.columns[np.asarray(self.support_)]
            return X.loc[:, cols]

        return X[:, self.support_]


class Chi2ThresholdSelector(BaseEstimator, TransformerMixin):
    """
    Selecciona variables con chi2 >= threshold.
    Requiere X no negativa.
    """

    def __init__(self, threshold=0.0):
        self.threshold = threshold
        self.support_ = None
        self.chi2_scores_ = None
        self.pvalues_ = None
        self.feature_names_in_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.to_list()
            X_fit = X.values
        else:
            self.feature_names_in_ = None
            X_fit = X

        self.chi2_scores_, self.pvalues_ = chi2(X_fit, y)
        self.support_ = self.chi2_scores_ >= self.threshold

        # si no sobrevive ninguna, dejar la mejor
        if not np.any(self.support_):
            best_idx = np.argmax(self.chi2_scores_)
            self.support_[best_idx] = True

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            cols = X.columns[np.asarray(self.support_)]
            return X.loc[:, cols]

        return X[:, self.support_]


# =========================================================
# Helpers
# =========================================================

def build_pipeline(
    estimator,
    selector_name,
    spearman_threshold=0.9,
    mi_threshold=0.01,
    chi2_threshold=3.84,
    random_state=42,
):
    if selector_name == "spearman_corr":
        return Pipeline([
            ("selector", SpearmanCorrelationDiscard(threshold=spearman_threshold)),
            ("scaler", StandardScaler()),
            ("model", clone(estimator)),
        ])

    elif selector_name == "mutual_info":
        return Pipeline([
            ("selector", MIThresholdSelector(
                threshold=mi_threshold,
                random_state=random_state
            )),
            ("scaler", StandardScaler()),
            ("model", clone(estimator)),
        ])

    elif selector_name == "chi2":
        return Pipeline([
            ("minmax", MinMaxScaler()),
            ("selector", Chi2ThresholdSelector(threshold=chi2_threshold)),
            ("scaler", StandardScaler()),
            ("model", clone(estimator)),
        ])

    else:
        raise ValueError(f"Selector no soportado: {selector_name}")


def get_scores_multiclass(fitted_pipe, X_part):
    """
    Devuelve scores continuos por clase:
    - predict_proba si existe
    - si no, decision_function si existe
    - si no, None
    """
    if hasattr(fitted_pipe, "predict_proba"):
        try:
            proba = fitted_pipe.predict_proba(X_part)
            if isinstance(proba, np.ndarray) and proba.ndim == 2:
                return proba
        except Exception:
            pass

    if hasattr(fitted_pipe, "decision_function"):
        try:
            dec = fitted_pipe.decision_function(X_part)
            if isinstance(dec, np.ndarray) and dec.ndim == 2:
                return dec
        except Exception:
            pass

    return None


def auc_multiclass_ovr(y_true, scores, average="macro"):
    if scores is None:
        return np.nan
    try:
        return roc_auc_score(y_true, scores, multi_class="ovr", average=average)
    except Exception:
        return np.nan


def compute_metrics(y_true, y_pred, average="macro", auc_value=np.nan):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average=average, zero_division=0),
        "Recall": recall_score(y_true, y_pred, average=average, zero_division=0),
        "F1": f1_score(y_true, y_pred, average=average, zero_division=0),
        "AUC": auc_value,
    }


def summarize(metrics_list, suffix="CV", decimals=4):
    df = pd.DataFrame(metrics_list)
    mean = df.mean(numeric_only=True)
    std = df.std(ddof=1, numeric_only=True)

    return {
        f"{c}_{suffix}": f"{mean[c]:.{decimals}f} ± {std[c]:.{decimals}f}"
        for c in mean.index
    }


# =========================================================
# Función principal
# =========================================================

def evaluate_models_oof_and_test_with_fs(
    X_df: pd.DataFrame,
    y_df: pd.DataFrame,
    models: dict,
    outer_splits: int = 10,
    inner_splits: int = 10,
    test_size: float = 0.2,
    random_state: int = 42,
    average: str = "macro",
    decimals: int = 4,
    selectors: list = None,
    spearman_threshold: float = 0.9,
    mi_threshold: float = 0.01,
    chi2_threshold: float = 3.84,
):
    """
    Evalúa múltiples modelos usando:
      - Outer CV: StratifiedShuffleSplit
      - Inner CV: StratifiedKFold para OOF en train

    Para cada modelo y cada selector genera una fila.

    Selectores soportados:
      - "spearman_corr"
      - "mutual_info"
      - "chi2"
    """

    if selectors is None:
        selectors = ["spearman_corr", "mutual_info", "chi2"]

    X = X_df.copy()
    y = y_df.iloc[:, 0].to_numpy()

    outer = StratifiedShuffleSplit(
        n_splits=outer_splits,
        test_size=test_size,
        random_state=random_state
    )

    all_rows = []

    for model_name, estimator in models.items():
        for selector_name in selectors:
            print(f"Evaluating {model_name} + {selector_name}...")

            test_metrics_all = []
            cv_metrics_all = []

            for train_idx, test_idx in outer.split(X, y):
                X_train = X.iloc[train_idx].copy()
                X_test = X.iloc[test_idx].copy()
                y_train = y[train_idx]
                y_test = y[test_idx]

                # -----------------------------
                # OOF en TRAIN (inner CV)
                # -----------------------------
                inner = StratifiedKFold(
                    n_splits=inner_splits,
                    shuffle=True,
                    random_state=random_state
                )

                oof_pred = np.empty_like(y_train)
                oof_scores = None

                for tr_idx, val_idx in inner.split(X_train, y_train):
                    X_tr = X_train.iloc[tr_idx].copy()
                    X_val = X_train.iloc[val_idx].copy()
                    y_tr = y_train[tr_idx]
                    y_val = y_train[val_idx]

                    pipe = build_pipeline(
                        estimator=estimator,
                        selector_name=selector_name,
                        spearman_threshold=spearman_threshold,
                        mi_threshold=mi_threshold,
                        chi2_threshold=chi2_threshold,
                        random_state=random_state,
                    )

                    pipe.fit(X_tr, y_tr)

                    # predicción clase OOF
                    oof_pred[val_idx] = pipe.predict(X_val)

                    # scores OOF para AUC
                    fold_scores = get_scores_multiclass(pipe, X_val)
                    if fold_scores is not None:
                        if oof_scores is None:
                            oof_scores = np.full(
                                (len(y_train), fold_scores.shape[1]),
                                np.nan,
                                dtype=float
                            )
                        oof_scores[val_idx, :] = fold_scores

                auc_oof = auc_multiclass_ovr(y_train, oof_scores, average=average)
                cv_metrics_all.append(
                    compute_metrics(
                        y_true=y_train,
                        y_pred=oof_pred,
                        average=average,
                        auc_value=auc_oof
                    )
                )

                # -----------------------------
                # TEST externo
                # -----------------------------
                pipe_full = build_pipeline(
                    estimator=estimator,
                    selector_name=selector_name,
                    spearman_threshold=spearman_threshold,
                    mi_threshold=mi_threshold,
                    chi2_threshold=chi2_threshold,
                    random_state=random_state,
                )

                pipe_full.fit(X_train, y_train)

                test_pred = pipe_full.predict(X_test)
                test_scores = get_scores_multiclass(pipe_full, X_test)
                auc_test = auc_multiclass_ovr(y_test, test_scores, average=average)

                test_metrics_all.append(
                    compute_metrics(
                        y_true=y_test,
                        y_pred=test_pred,
                        average=average,
                        auc_value=auc_test
                    )
                )

            row = {
                "Model": model_name,
                "Feature_Selection": selector_name,
            }
            row.update(summarize(test_metrics_all, suffix="Testing", decimals=decimals))
            row.update(summarize(cv_metrics_all, suffix="CV", decimals=decimals))

            all_rows.append(row)

    df_final_summary = pd.DataFrame(all_rows)[[
        "Model",
        "Feature_Selection",
        "Accuracy_Testing",
        "Precision_Testing",
        "Recall_Testing",
        "F1_Testing",
        "AUC_Testing",
        "Accuracy_CV",
        "Precision_CV",
        "Recall_CV",
        "F1_CV",
        "AUC_CV",
    ]]

    return df_final_summary

## HY3

In [23]:
X_HY3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY3_full.csv", index_col=0)
y_HY3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_full.csv", index_col=0)
print("X_HY3_data shape:", X_HY3_data.shape)
print("y_HY3_data shape:", y_HY3_data.shape)

X_HY3_data shape: (909, 931)
y_HY3_data shape: (909, 1)


In [ ]:
HY3_fe_results = evaluate_models_oof_and_test_with_fs(
    X_df=X_HY3_data,
    y_df=y_HY3_data,
    models=classification_models,
    outer_splits=10,
    inner_splits=10,
    test_size=0.2,
    random_state=42,
    average="macro",
    selectors=["spearman_corr", "mutual_info", "chi2"],
    spearman_threshold=0.9,
    mi_threshold=0.01,
    chi2_threshold=3.84,
)
HY3_fe_results.head(10)

Evaluating decision_tree + spearman_corr...
Evaluating decision_tree + mutual_info...
Evaluating decision_tree + chi2...
Evaluating random_forest + spearman_corr...
Evaluating random_forest + mutual_info...
Evaluating random_forest + chi2...
Evaluating extra_trees + spearman_corr...
Evaluating extra_trees + mutual_info...
Evaluating extra_trees + chi2...
Evaluating xgboost + spearman_corr...
Evaluating xgboost + mutual_info...
Evaluating xgboost + chi2...
Evaluating adaboost + spearman_corr...
Evaluating adaboost + mutual_info...
Evaluating adaboost + chi2...
Evaluating logistic_regression + spearman_corr...
Evaluating logistic_regression + mutual_info...
Evaluating logistic_regression + chi2...


/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/st

Evaluating sgd + spearman_corr...


/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))


Evaluating sgd + mutual_info...
Evaluating sgd + chi2...
Evaluating svc + spearman_corr...
Evaluating svc + mutual_info...
Evaluating svc + chi2...
Evaluating knn + spearman_corr...
Evaluating knn + mutual_info...
Evaluating knn + chi2...
Evaluating gaussian_nb + spearman_corr...
Evaluating gaussian_nb + mutual_info...
Evaluating gaussian_nb + chi2...


,Model,Feature_Selection,Accuracy_Testing,Precision_Testing,Recall_Testing,F1_Testing,AUC_Testing,Accuracy_CV,Precision_CV,Recall_CV,F1_CV,AUC_CV
0,decision_tree,spearman_corr,0.8473 ± 0.0193,0.6201 ± 0.0498,0.6317 ± 0.0600,0.6235 ± 0.0520,0.7689 ± 0.0332,0.8250 ± 0.0107,0.6257 ± 0.0353,0.6288 ± 0.0363,0.6265 ± 0.0354,0.7606 ± 0.0206
1,decision_tree,mutual_info,0.8445 ± 0.0206,0.6452 ± 0.0366,0.6425 ± 0.0327,0.6410 ± 0.0301,0.7735 ± 0.0207,0.8322 ± 0.0081,0.6473 ± 0.0196,0.6512 ± 0.0203,0.6482 ± 0.0184,0.7740 ± 0.0107
2,decision_tree,chi2,0.8209 ± 0.0282,0.6013 ± 0.0522,0.6009 ± 0.0426,0.5998 ± 0.0458,0.7447 ± 0.0288,0.8343 ± 0.0155,0.6615 ± 0.0338,0.6587 ± 0.0398,0.6581 ± 0.0358,0.7779 ± 0.0228
3,random_forest,spearman_corr,0.8978 ± 0.0196,0.5990 ± 0.0132,0.6166 ± 0.0137,0.6074 ± 0.0133,0.9359 ± 0.0242,0.8990 ± 0.0046,0.6002 ± 0.0029,0.6174 ± 0.0033,0.6086 ± 0.0030,0.9532 ± 0.0062
4,random_forest,mutual_info,0.9038 ± 0.0145,0.6697 ± 0.1413,0.6333 ± 0.0295,0.6333 ± 0.0479,0.9473 ± 0.0156,0.8986 ± 0.0047,0.6917 ± 0.0904,0.6297 ± 0.0166,0.6310 ± 0.0273,0.9536 ± 0.0083
5,random_forest,chi2,0.8984 ± 0.0111,0.6327 ± 0.1060,0.6232 ± 0.0219,0.6187 ± 0.0359,0.9419 ± 0.0186,0.8982 ± 0.0049,0.7526 ± 0.0809,0.6420 ± 0.0176,0.6525 ± 0.0292,0.9568 ± 0.0046
6,extra_trees,spearman_corr,0.9016 ± 0.0177,0.6016 ± 0.0118,0.6191 ± 0.0121,0.6100 ± 0.0119,0.9413 ± 0.0185,0.8993 ± 0.0043,0.6001 ± 0.0028,0.6173 ± 0.0031,0.6085 ± 0.0029,0.9529 ± 0.0045
7,extra_trees,mutual_info,0.9022 ± 0.0124,0.6354 ± 0.1038,0.6256 ± 0.0203,0.6213 ± 0.0339,0.9449 ± 0.0161,0.8961 ± 0.0064,0.7472 ± 0.1206,0.6338 ± 0.0225,0.6403 ± 0.0356,0.9568 ± 0.0063
8,extra_trees,chi2,0.9000 ± 0.0146,0.6006 ± 0.0101,0.6177 ± 0.0102,0.6089 ± 0.0100,0.9439 ± 0.0180,0.8989 ± 0.0041,0.7723 ± 0.0845,0.6375 ± 0.0119,0.6460 ± 0.0201,0.9546 ± 0.0066
9,xgboost,spearman_corr,0.9044 ± 0.0158,0.7794 ± 0.1737,0.6593 ± 0.0397,0.6742 ± 0.0608,0.9095 ± 0.0213,0.8975 ± 0.0068,0.8054 ± 0.0738,0.6794 ± 0.0282,0.7048 ± 0.0373,0.9394 ± 0.0110


## HY4

In [25]:
X_HY4_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)
y_HY4_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_full.csv", index_col=0)
print("X_HY4_data shape:", X_HY4_data.shape)
print("y_HY4_data shape:", y_HY4_data.shape)

X_HY4_data shape: (909, 931)
y_HY4_data shape: (909, 1)


In [26]:
HY4_fe_results = evaluate_models_oof_and_test_with_fs(
    X_df=X_HY4_data,
    y_df=y_HY4_data,
    models=classification_models,
    outer_splits=10,
    inner_splits=10,
    test_size=0.2,
    random_state=42,
    average="macro",
    selectors=["spearman_corr", "mutual_info", "chi2"],
    spearman_threshold=0.9,
    mi_threshold=0.01,
    chi2_threshold=3.84,
)
HY4_fe_results.head(10)

Evaluating decision_tree + spearman_corr...
Evaluating decision_tree + mutual_info...
Evaluating decision_tree + chi2...
Evaluating random_forest + spearman_corr...
Evaluating random_forest + mutual_info...
Evaluating random_forest + chi2...
Evaluating extra_trees + spearman_corr...
Evaluating extra_trees + mutual_info...


KeyboardInterrupt: 

## MCID

In [ ]:
X_MCID_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_full.csv", index_col=0)
y_MCID_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_full.csv", index_col=0)
print("X_MCID_data shape:", X_MCID_data.shape)
print("y_MCID_data shape:", y_MCID_data.shape)

In [ ]:
MCID_fe_results = evaluate_models_oof_and_test_with_fs(
    X_df=X_MCID_data,
    y_df=y_MCID_data,
    models=classification_models,
    outer_splits=10,
    inner_splits=10,
    test_size=0.2,
    random_state=42,
    average="macro",
    selectors=["spearman_corr", "mutual_info", "chi2"],
    spearman_threshold=0.9,
    mi_threshold=0.01,
    chi2_threshold=3.84,
)
MCID_fe_results.head(10)

---

# AGNOSTIC FEATURE SELECTION MOTOR
- Variance treshold
- Correlation de spearman
- MI
- Chi2

## HY3

In [ ]:
cols=datos['M_data']
remove_cols = ['NHY_mean', 'NHY_min', 'NHY_max', 'NHY_var', 'NHY_std']
valid_cols = [x for x in cols if x not in remove_cols]
X_HY3_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY3_full.csv", index_col=0)[valid_cols]
y_HY3_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_full.csv", index_col=0)
print("X_HY3_motor_data shape:", X_HY3_motor_data.shape)
print("y_HY3_motor_data shape:", y_HY3_motor_data.shape)

In [ ]:
HY3_fe_MOTOR_results = evaluate_models_oof_and_test_with_fs(
    X_df=X_HY3_motor_data,
    y_df=y_HY3_motor_data,
    models=classification_models,
    outer_splits=10,
    inner_splits=10,
    test_size=0.2,
    random_state=42,
    average="macro",
    selectors=["spearman_corr", "mutual_info", "chi2"],
    spearman_threshold=0.9,
    mi_threshold=0.01,
    chi2_threshold=3.84,
)
HY3_fe_MOTOR_results.head(30)

## HY4

In [ ]:
cols=datos['M_data']
remove_cols = ['NHY_mean', 'NHY_min', 'NHY_max', 'NHY_var', 'NHY_std']
valid_cols = [x for x in cols if x not in remove_cols]
X_HY4_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)[valid_cols]
y_HY4_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_full.csv", index_col=0)
print("X_HY4_motor_data shape:", X_HY4_motor_data.shape)
print("y_HY4_motor_data shape:", y_HY4_motor_data.shape)

In [ ]:
HY4_fe_MOTOR_results = evaluate_models_oof_and_test_with_fs(
    X_df=X_HY4_motor_data,
    y_df=y_HY4_motor_data,
    models=classification_models,
    outer_splits=10,
    inner_splits=10,
    test_size=0.2,
    random_state=42,
    average="macro",
    selectors=["spearman_corr", "mutual_info", "chi2"],
    spearman_threshold=0.9,
    mi_threshold=0.01,
    chi2_threshold=3.84,
)
HY4_fe_MOTOR_results.head(30)

## MCID

In [ ]:
cols=datos['M_data']
X_MCID_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_full.csv", index_col=0)[cols]
y_MCID_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_full.csv", index_col=0)
print("X_MCID_motor_data shape:", X_MCID_motor_data.shape)
print("y_MCID_motor_data shape:", y_MCID_motor_data.shape)

In [ ]:
MCID_fe_MOTOR_results = evaluate_models_oof_and_test_with_fs(
    X_df=X_MCID_motor_data,
    y_df=y_MCID_motor_data,
    models=classification_models,
    outer_splits=10,
    inner_splits=10,
    test_size=0.2,
    random_state=42,
    average="macro",
    selectors=["spearman_corr", "mutual_info", "chi2"],
    spearman_threshold=0.9,
    mi_threshold=0.01,
    chi2_threshold=3.84,
)
MCID_fe_MOTOR_results.head(30)

---

# AGNOSTIC FEATURE SELECTION NON MOTOR
- Variance treshold
- Correlation de spearman
- MI
- Chi2

## HY3

In [ ]:
cols=datos['NM_data']
X_HY3_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY3_full.csv", index_col=0)[cols]
y_HY3_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_full.csv", index_col=0)
print("X_HY3_NO_motor_data shape:", X_HY3_NO_motor_data.shape)
print("y_HY3_NO_motor_data shape:", y_HY3_NO_motor_data.shape)

In [ ]:
HY3_fe_NO_MOTOR_results = evaluate_models_oof_and_test_with_fs(
    X_df=X_HY3_NO_motor_data,
    y_df=y_HY3_NO_motor_data,
    models=classification_models,
    outer_splits=10,
    inner_splits=10,
    test_size=0.2,
    random_state=42,
    average="macro",
    selectors=["spearman_corr", "mutual_info", "chi2"],
    spearman_threshold=0.9,
    mi_threshold=0.01,
    chi2_threshold=3.84,
)
HY3_fe_NO_MOTOR_results.head(30)

## HY4

In [ ]:
cols=datos['NM_data']
X_HY4_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)[cols]
y_HY4_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_full.csv", index_col=0)
print("X_HY4_NO_motor_data shape:", X_HY4_NO_motor_data.shape)
print("y_HY4_NO_motor_data shape:", y_HY4_NO_motor_data.shape)

In [ ]:
HY4_fe_NO_MOTOR_results = evaluate_models_oof_and_test_with_fs(
    X_df=X_HY4_NO_motor_data,
    y_df=y_HY4_NO_motor_data,
    models=classification_models,
    outer_splits=10,
    inner_splits=10,
    test_size=0.2,
    random_state=42,
    average="macro",
    selectors=["spearman_corr", "mutual_info", "chi2"],
    spearman_threshold=0.9,
    mi_threshold=0.01,
    chi2_threshold=3.84,
)
HY4_fe_NO_MOTOR_results.head(30)

## MCID

In [ ]:
cols=datos['NM_data']
X_MCID_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_full.csv", index_col=0)[cols]
y_MCID_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_full.csv", index_col=0)
print("X_MCID_NO_motor_data shape:", X_MCID_NO_motor_data.shape)
print("y_MCID_NO_motor_data shape:", y_MCID_NO_motor_data.shape)

In [ ]:
MCID_fe_NO_MOTOR_results = evaluate_models_oof_and_test_with_fs(
    X_df=X_MCID_NO_motor_data,
    y_df=y_MCID_NO_motor_data,
    models=classification_models,
    outer_splits=10,
    inner_splits=10,
    test_size=0.2,
    random_state=42,
    average="macro",
    selectors=["spearman_corr", "mutual_info", "chi2"],
    spearman_threshold=0.9,
    mi_threshold=0.01,
    chi2_threshold=3.84,
)
MCID_fe_NO_MOTOR_results.head(30)

---